# sheet_operate — Phase 5 匯出（merge LoRA → GGUF q4_K_M）
把 GRPO 訓完的 adapter 合併回 base model，量化成 GGUF 存到 Drive。
下載到本機後即可用 Ollama（`ollama create`）或 llama-cpp-python 直接推論。


In [ ]:
CFG = dict(
    drive_root = "/content/drive/MyDrive/公司/AI ML Research/未命名資料夾/sheet_operate",
    adapter    = "ckpt_grpo_v1/checkpoint-400",   # GRPO 權重在 checkpoint（adapter_grpo_v1 未曾存出）
    base_model = "Qwen/Qwen3-4B-Instruct-2507",
    quant      = "q8_0",                # 官方轉換器免編譯支援；4.3GB，8GB 卡可跑且品質優於 q4
    lora_r = 64, lora_alpha = 64,
)

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CFG["base_model"], max_seq_length = 8192,
    dtype = None, load_in_4bit = False,
)
model = FastLanguageModel.get_peft_model(
    model, r = CFG["lora_r"], lora_alpha = CFG["lora_alpha"], lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict
import glob

# 自動搜尋 adapter：先試 CFG 路徑，再掃同層兄弟資料夾（防資料夾名打字差異）
def find_adapter():
    p = os.path.join(CFG["drive_root"], CFG["adapter"], "adapter_model.safetensors")
    if os.path.exists(p):
        return p
    parent = os.path.dirname(CFG["drive_root"].rstrip("/"))
    for root in sorted(glob.glob(os.path.join(parent, "*"))):
        p = os.path.join(root, CFG["adapter"], "adapter_model.safetensors")
        if os.path.exists(p):
            CFG["drive_root"] = root      # 輸出的 gguf 也存到 adapter 所在資料夾
            return p
    raise AssertionError(f"{parent} 底下找不到 {CFG['adapter']}；該層內容：{os.listdir(parent)}")

adapter_path = find_adapter()
print("使用 adapter：", adapter_path)
set_peft_model_state_dict(model, load_file(adapter_path))

In [ ]:
# 合併 16bit → llama.cpp 官方轉換器 → GGUF（免編譯路徑，比 unsloth 內建轉換穩定）
model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")

import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/ggml-org/llama.cpp"], check=False)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gguf", "sentencepiece"], check=True)
subprocess.run([sys.executable, "llama.cpp/convert_hf_to_gguf.py", "merged_model",
                "--outfile", f"sheetops-{CFG['quant']}.gguf",
                "--outtype", CFG["quant"]], check=True)

import shutil
dst = os.path.join(CFG["drive_root"], f"sheetops-{CFG['quant']}.gguf")
shutil.copy(f"sheetops-{CFG['quant']}.gguf", dst)
print(f"已複製：{dst}（{os.path.getsize(dst)/1e9:.2f} GB）")
print("強制上傳中（4.3GB 需數分鐘，跑完才算真正存進 Drive）…")
drive.flush_and_unmount()
print("✅ 上傳完成——現在可以去 Drive 網頁下載了")

## 下載後的本機步驟（Windows）
1. 從 Drive 下載 `sheetops-q4_k_m.gguf` 到 repo 的 `deploy/` 資料夾
2. `ollama create sheetops -f deploy/Modelfile`
3. 開始用：`python scripts/sheetops_cli.py 報表.xlsx "你的指令"`
   （或不裝 Ollama：`--gguf deploy/sheetops-q4_k_m.gguf` 直連推論）
